# Populate MongoDb with Customer & Article Data 

## I. Retrieve Customer Data

### Customer Account

In [1]:
import pandas as pd

In [2]:
customers_file_path = "/data/customers.csv"

customers_df = pd.read_csv(customers_file_path)
customers_df.head(5)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [3]:
customers_df.shape

(1371980, 7)

In [4]:
customers_df.isna().sum()

customer_id                    0
FN                        895050
Active                    907576
club_member_status          6062
fashion_news_frequency     16011
age                        15861
postal_code                    0
dtype: int64

In [5]:
print(f"Average Customer Age is: {customers_df['age'].mean().round(0)}")
print("")

Average Customer Age is: 36.0



In [6]:
customers_df['Active'].unique()

array([nan,  1.])

In [7]:
customers_df['club_member_status'].unique()

array(['ACTIVE', nan, 'PRE-CREATE', 'LEFT CLUB'], dtype=object)

In [8]:
customers_df['fashion_news_frequency'].unique()

array(['NONE', 'Regularly', nan, 'Monthly'], dtype=object)

In [9]:
customers_df['age'].fillna(customers_df['age'].mean().round(0), inplace=True)
customers_df['Active'].fillna(0, inplace=True)
customers_df['club_member_status'].fillna('NONE', inplace=True)
customers_df['fashion_news_frequency'].fillna('NONE', inplace=True)
customers_df.drop(columns=['FN', 'postal_code'], inplace=True)

/tmp/ipykernel_23824/236250504.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  customers_df['age'].fillna(customers_df['age'].mean().round(0), inplace=True)
/tmp/ipykernel_23824/236250504.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].metho

In [10]:
customers_df.columns

Index(['customer_id', 'Active', 'club_member_status', 'fashion_news_frequency',
       'age'],
      dtype='object')

### Customer Purchase History

In [11]:
purchase_history = pd.read_csv('/data/transactions_train.csv')
purchase_history.head(3)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2


In [12]:
purchase_history.isna().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

In [13]:
purchase_history.columns

Index(['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'], dtype='object')

In [14]:
from tqdm import tqdm

tqdm.pandas()

def prepare_customer_documents(customers_df: pd.DataFrame = customers_df, purchase_history: pd.DataFrame = purchase_history):
    """
    Aggregate customer data with purchase history for MongoDB ingestion.
    
    Args:
        customers_df: DataFrame with customer information
        purchase_history: DataFrame with purchase transactions
    
    Returns:
        List of dictionaries ready for MongoDB insertion
    """

    print("Creating Purchase History Groups")
    purchase_groups = purchase_history.groupby('customer_id')['article_id'].progress_apply(list).reset_index()
    purchase_groups.columns = ['customer_id', 'purchased_articles']

    print("Merging DataFrames")
    customer_documents = customers_df.merge(
        purchase_groups, 
        on='customer_id', 
        how='left'
    )
    
    print("Exporting Data to Dict")
    customer_documents['purchased_articles'] = customer_documents['purchased_articles'].progress_apply(
        lambda x: x if isinstance(x, list) else []
    )
    
    documents = customer_documents.to_dict('records')
    
    return documents


In [15]:
customer_documents = prepare_customer_documents()

Creating Purchase History Groups


100%|██████████| 1362281/1362281 [00:23<00:00, 57469.75it/s]


Merging DataFrames
Exporting Data to Dict


100%|██████████| 1371980/1371980 [00:00<00:00, 2310165.13it/s]


In [16]:
customer_documents

[{'customer_id': '00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657',
  'Active': 0.0,
  'club_member_status': 'ACTIVE',
  'fashion_news_frequency': 'NONE',
  'age': 49.0,
  'purchased_articles': [625548001,
   176209023,
   627759010,
   697138006,
   568601006,
   568601006,
   607642008,
   745232001,
   656719005,
   797065001,
   797065001,
   785186005,
   694736004,
   785710001,
   812683013,
   841260003,
   887593002,
   890498002,
   795440001,
   859416011,
   568601043]},
 {'customer_id': '0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa',
  'Active': 0.0,
  'club_member_status': 'ACTIVE',
  'fashion_news_frequency': 'NONE',
  'age': 25.0,
  'purchased_articles': [583558001,
   639677008,
   640244003,
   521269001,
   666448006,
   583558001,
   673677002,
   666448006,
   690229003,
   723469005,
   640174001,
   700515001,
   614854003,
   724904001,
   723529001,
   640021012,
   351484002,
   658298001,
   734282002,
   640174001,
   640

In [17]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

env_filepath = '/workspace/.env'
load_dotenv(env_filepath)

MONGO_CONNECTION_STRING = os.getenv('MONGO_CONNECTION_STRING')

def ingest_to_mongodb(documents: list, connection_string: str = MONGO_CONNECTION_STRING,
                    database_name: str = 'stylistai_db', collection_name: str = 'customers',
                    batch_size: int =1000):
    """
    Ingest customer documents into MongoDB.
    
    Args:
        documents: List of customer dictionaries
        connection_string: MongoDB connection string
        database_name: Name of the database
        collection_name: Name of the collection
    """
    
    client = MongoClient(connection_string)
    db = client[database_name]
    collection = db[collection_name]
    
    total_docs = len(documents)
    inserted_count = 0
    
    print(f"Starting ingestion of {total_docs} documents...")
    print(f"Batch size: {batch_size}")
    print("-" * 50)
    
    # Insert in batches with progress tracking
    for i in range(0, total_docs, batch_size):
        batch = documents[i:i + batch_size]
        
        try:
            result = collection.insert_many(batch, ordered=False)
            inserted_count += len(result.inserted_ids)
            
            # Calculate and display progress
            progress = (inserted_count / total_docs) * 100
            print(f"Progress: {inserted_count}/{total_docs} ({progress:.1f}%) - Batch {i//batch_size + 1}")
            
        except Exception as e:
            print(f"Error inserting batch {i//batch_size + 1}: {e}")
            continue
    
    print("-" * 50)
    print(f"✓ Ingestion complete: {inserted_count}/{total_docs} documents inserted")
    
    print("Creating index on customer_id...")
    collection.create_index("customer_id", unique=True)
    print("✓ Index created successfully")
    
    client.close()


In [18]:
ingest_to_mongodb(documents=customer_documents)

Starting ingestion of 1371980 documents...
Batch size: 1000
--------------------------------------------------
Progress: 1000/1371980 (0.1%) - Batch 1
Progress: 2000/1371980 (0.1%) - Batch 2
Progress: 3000/1371980 (0.2%) - Batch 3
Progress: 4000/1371980 (0.3%) - Batch 4
Progress: 5000/1371980 (0.4%) - Batch 5
Progress: 6000/1371980 (0.4%) - Batch 6
Progress: 7000/1371980 (0.5%) - Batch 7
Progress: 8000/1371980 (0.6%) - Batch 8
Progress: 9000/1371980 (0.7%) - Batch 9
Progress: 10000/1371980 (0.7%) - Batch 10
Progress: 11000/1371980 (0.8%) - Batch 11
Progress: 12000/1371980 (0.9%) - Batch 12
Progress: 13000/1371980 (0.9%) - Batch 13
Progress: 14000/1371980 (1.0%) - Batch 14
Progress: 15000/1371980 (1.1%) - Batch 15
Progress: 16000/1371980 (1.2%) - Batch 16
Progress: 17000/1371980 (1.2%) - Batch 17
Progress: 18000/1371980 (1.3%) - Batch 18
Progress: 19000/1371980 (1.4%) - Batch 19
Progress: 20000/1371980 (1.5%) - Batch 20
Progress: 21000/1371980 (1.5%) - Batch 21
Progress: 22000/1371980 (

## II. Get Article Metadata

In [19]:
import pandas as pd

df = pd.read_csv('/data/articles.csv')
df.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [20]:
df.columns

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')

## III. Customer Purchase History

## IV. Populate Mongo Instance